# 02 — Traditional ML benchmark: AA-identity baseline (TEM-1 / Firnberg)

## Why an AA-identity baseline

This notebook establishes the **no-language-model control** for the project's central question — can
protein-language-model features predict whether a TEM-1 variant is functional? Before crediting any
pLLM feature with predictive power, we need to know what a model can achieve using **amino-acid identity
alone**: which residue changed, to what, and where — with no evolutionary, structural, or language-model
signal. That is the floor any pLLM-feature model must beat to justify its added complexity (lit-review
decision D027: bounding baselines are *measured*, not asserted).

Two design choices make this baseline informative rather than a formality:

- **Region-holdout splits.** Under the contiguous split, whole stretches of the protein are withheld, so
  an identity-only model cannot have seen the held-out positions — it can only transfer *substitution
  chemistry* (e.g. "proline is usually disruptive") learned elsewhere. The gap between the random split
  (easy) and the contiguous split (hard) quantifies how much apparent skill is just position memorization.
- **It calibrates later claims.** If identity alone already predicts functionality well, that tempers
  claims about pLLM features; if it collapses on the hard split, that is the concrete motivation for the
  pLLM approach.

**Positive class = functional (`DMS_score_bin == 1`).** Models: L2 logistic regression, random forest,
XGBoost, RBF SVM, plus a majority-class floor (lit-review D023–D027). Splits: random, modulo, contiguous.
Evaluation: ROC-AUC, PR-AUC, accuracy, balanced accuracy, F1, MCC, and a utility score (below), with
bootstrap CIs, DeLong and McNemar tests, and SHAP attribution for all four models.

In [ ]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, os, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix, roc_curve,
    precision_recall_curve)
from xgboost import XGBClassifier
import joblib

SEED = 42; N_SPLITS = 5; N_BOOT = 2000
SHAP_N_EXPLAIN = 200      # points explained per model (stratified)
SHAP_N_BACKGROUND = 50    # k-means background for KernelSHAP (svm, logreg)
SHAP_SPLIT = "random"     # attribution computed on this split's OOF model
AA = list("ACDEFGHIKLMNPQRSTVWY")
PAL = {"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","teal":"#2a9d8f"}
MODEL_COLORS = {"logreg":PAL["blue"],"rf":PAL["green"],"xgboost":PAL["purple"],
                "svm":PAL["pink"],"dummy":"#9aa0a6"}
sns.set_theme(style="whitegrid")
NBNAME = "02_traditional_ml_aa_identity_benchmark"
FIGDIR = FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True)
MODELDIR = MODELS/"traditional_ml_aa_identity"; MODELDIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| seed:", SEED)

## 1. Load the modeling data

Read the processed AA-identity table (built in the dataset step). Validate at the boundary.

In [ ]:
df = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
assert {"position","wt_aa","mut_aa","DMS_score_bin"} <= set(df.columns)
assert df.DMS_score_bin.isin([0,1]).all() and df.notna().all().all()
print("rows:", len(df), "| positions:", df.position.nunique(),
      "| class balance:", df.DMS_score_bin.value_counts().to_dict())
df.head(3)

## 2. Feature matrix — mutation-descriptor one-hot

AA identity only: one-hot of **position** + **wild-type residue** + **substituted residue**. No engineered
physicochemical, evolutionary, or structural features. The leakage check confirms no column is derived
from the label.

In [ ]:
enc = OneHotEncoder(categories=[sorted(df.position.unique()), AA, AA],
                    handle_unknown="ignore", sparse_output=False, dtype=np.float32)
X = enc.fit_transform(df[["position","wt_aa","mut_aa"]])
feat_names = enc.get_feature_names_out()
y = df.DMS_score_bin.values.astype(int)
pos = df.position.values

# leakage guard (CLAUDE.md): no label-derived column; nothing perfectly tracks y
assert not any(("DMS" in f or "bin" in f or "score" in f) for f in feat_names)
mx = max(abs(np.corrcoef(X[:,j],y)[0,1]) for j in range(X.shape[1]) if X[:,j].std()>0)
assert mx < 0.999, "leakage: a feature perfectly tracks the label"
print(f"X: {X.shape}  ({sum(f.startswith('position_') for f in feat_names)} position + "
      f"{sum(f.startswith('wt_aa_') for f in feat_names)} wt + "
      f"{sum(f.startswith('mut_aa_') for f in feat_names)} mut)")
print(f"max |corr(feature, y)| = {mx:.3f}  -> no leakage")

## 3. Three split schemes

- **random** — stratified k-fold on rows; easiest, optimistic (positions appear in both train and test).
- **modulo** — hold out positions where `position % k == fold`; scattered residues withheld, no position
  leaks across train/test.
- **contiguous** — hold out contiguous blocks of positions; whole regions unseen. Hardest and most
  realistic: the model must generalize substitution effects to sequence regions it never saw.

In [ ]:
def make_folds(df, n_splits=N_SPLITS, seed=SEED):
    idx = np.arange(len(df)); pos = df.position.values; folds={}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds["random"] = list(skf.split(idx, df.DMS_score_bin.values))
    uniq = np.sort(df.position.unique())
    folds["modulo"] = [(idx[~np.isin(pos, uniq[uniq%n_splits==k])],
                        idx[ np.isin(pos, uniq[uniq%n_splits==k])]) for k in range(n_splits)]
    folds["contiguous"] = [(idx[~np.isin(pos,b)], idx[np.isin(pos,b)])
                           for b in np.array_split(uniq, n_splits)]
    return folds

folds = make_folds(df)
for scheme in ["modulo","contiguous"]:
    for k,(tr,te) in enumerate(folds[scheme]):
        assert not (set(pos[tr]) & set(pos[te])), f"{scheme} fold {k} leaks a position"
for scheme,fl in folds.items():
    tested=np.concatenate([te for _,te in fl])
    assert len(np.unique(tested))==len(df)
    print(f"{scheme:11s} test sizes {[len(te) for _,te in fl]}")
print("no position leakage in modulo/contiguous; full coverage in all schemes")

## 4. Models and evaluation

Five estimators (four learners + a majority-class floor). Continuous features are standardized inside a
pipeline for the scale-sensitive learners (logreg, SVM); trees are scale-invariant.

**Utility score.** Beyond standard metrics we report a domain-weighted score. With *positive = functional*:
a predicted-functional variant that is truly functional (**TP**) and a predicted-nonfunctional variant that
is truly nonfunctional (**TN**) are both good outcomes (**+1**); predicting nonfunctional when it is actually
functional (**FN**) is a tolerable miss (**−0.25**); predicting functional when it is actually nonfunctional
(**FP**) is the costly error (**−1**). Mean utility per prediction lies in [−1, 1].

In [ ]:
UTIL = {"TP":1.0,"TN":1.0,"FN":-0.25,"FP":-1.0}
def utility(y_true,y_pred):
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    return (UTIL["TP"]*tp+UTIL["TN"]*tn+UTIL["FN"]*fn+UTIL["FP"]*fp)/len(y_true)

def metrics(y_true, proba, thr=0.5):
    pred=(proba>=thr).astype(int)
    return dict(roc_auc=roc_auc_score(y_true,proba), pr_auc=average_precision_score(y_true,proba),
        accuracy=accuracy_score(y_true,pred), balanced_acc=balanced_accuracy_score(y_true,pred),
        f1=f1_score(y_true,pred), mcc=matthews_corrcoef(y_true,pred), utility=utility(y_true,pred))

def make_models():
    return {
        "logreg":  lambda: make_pipeline(StandardScaler(with_mean=False),
                        LogisticRegression(C=1.0, max_iter=5000, random_state=SEED)),
        "rf":      lambda: RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=SEED),
        "xgboost": lambda: XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1,
                        subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
                        tree_method="hist", random_state=SEED, n_jobs=-1),
        "svm":     lambda: make_pipeline(StandardScaler(with_mean=False),
                        CalibratedClassifierCV(SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED), ensemble=False)),
        "dummy":   lambda: DummyClassifier(strategy="most_frequent"),
    }
print("models:", list(make_models().keys()))
print("utility weights:", UTIL)

## 5. Train & evaluate — all models × all splits

Out-of-fold (OOF) predictions are collected across folds so every variant is predicted exactly once per
(model, split). Per-fold metrics give the mean ± std; OOF probabilities feed the ROC/PR curves and the
significance tests. Fitted estimators are retained for SHAP and for the saved artifacts.

In [ ]:
def run_all(X, y, folds, models):
    oof_proba, fold_metrics, fitted = {}, {}, {}
    for scheme, fl in folds.items():
        for mname, mk in models.items():
            key=(scheme,mname); oof=np.full(len(y), np.nan); per=[]
            for tr,te in fl:
                m=mk(); m.fit(X[tr], y[tr])
                p = m.predict_proba(X[te])[:,1] if hasattr(m,"predict_proba") else m.predict(X[te]).astype(float)
                oof[te]=p; per.append(metrics(y[te], p))
                if key not in fitted: fitted[key]=m   # keep first-fold model for inspection
            oof_proba[key]=oof
            fold_metrics[key]=pd.DataFrame(per)
    return oof_proba, fold_metrics, fitted

t0=time.time()
oof_proba, fold_metrics, fitted = run_all(X, y, folds, make_models())
print(f"trained {len(oof_proba)} (model x split) combos in {time.time()-t0:.0f}s")

In [ ]:
# assemble the mean +/- std metrics grid
rows=[]
for (scheme,mname),fm in fold_metrics.items():
    r={"split":scheme,"model":mname}
    for c in fm.columns:
        r[f"{c}_mean"]=fm[c].mean(); r[f"{c}_std"]=fm[c].std()
    rows.append(r)
grid=pd.DataFrame(rows).sort_values(["split","roc_auc_mean"], ascending=[True,False]).reset_index(drop=True)
disp_cols=["split","model","roc_auc_mean","pr_auc_mean","balanced_acc_mean","f1_mean","mcc_mean","utility_mean"]
from IPython.display import display
display(grid[disp_cols].round(3))

## 6. Statistical significance

Point estimates alone can mislead — a 0.01 AUC gap may be noise. Each test below answers a specific
question:

- **Bootstrap 95% CI** on ROC-AUC / PR-AUC: resample the OOF predictions (2000×) to put an uncertainty band
  on each metric. Overlapping CIs across models mean the apparent ranking may not be real.
- **DeLong test**: the standard test for whether two ROC-AUCs computed on the *same* samples differ
  significantly; it accounts for the correlation induced by shared test points, which a naive two-sample
  test ignores. Used here for every model vs the best model, per split.
- **McNemar's test**: the correct paired test for comparing two classifiers' *accuracy* on the same data —
  it looks only at the discordant predictions (one model right where the other is wrong). Used vs the best
  model per split.
- **Comparison vs the Dummy floor**: bootstrap p-value that each model's AUC exceeds 0.5-equivalent
  majority-class performance. A model that cannot clear this is not learning anything.

In [ ]:
def bootstrap_ci(y_true, proba, fn, n_boot=N_BOOT, seed=SEED):
    rng=np.random.default_rng(seed); n=len(y_true); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        if len(np.unique(y_true[idx]))<2: continue
        vals.append(fn(y_true[idx], proba[idx]))
    return float(np.mean(vals)), float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))

def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2

def delong_test(y_true, p1, p2):
    y=y_true.astype(int); posm=y==1; negm=y==0; m=posm.sum(); n=negm.sum()
    preds=np.vstack([p1,p2])
    tz=np.array([_midrank(p) for p in preds])
    auc=np.array([(stats.rankdata(np.concatenate([p[posm],p[negm]]))[:m].sum()-m*(m+1)/2)/(m*n) for p in preds])
    v01=(tz[:,y==1]-np.array([_midrank(p[posm]) for p in preds]))/n
    v10=1.0-(tz[:,y==0]-np.array([_midrank(p[negm]) for p in preds]))/m
    sx=np.cov(v01); sy=np.cov(v10); s=sx/m+sy/n
    var=s[0,0]+s[1,1]-2*s[0,1]
    if var<=0: return auc[0],auc[1],1.0
    z=(auc[0]-auc[1])/np.sqrt(var); return float(auc[0]),float(auc[1]),float(2*stats.norm.sf(abs(z)))

from statsmodels.stats.contingency_tables import mcnemar
def mcnemar_test(y_true, p1, p2, thr=0.5):
    a=(p1>=thr).astype(int); b=(p2>=thr).astype(int)
    n01=((a==y_true)&(b!=y_true)).sum(); n10=((a!=y_true)&(b==y_true)).sum()
    r=mcnemar([[0,n01],[n10,0]], exact=False, correction=True)
    return int(n01), int(n10), float(r.pvalue)
print("significance helpers defined")

In [ ]:
# CIs + significance vs the best model per split, and vs the dummy floor
learners=["logreg","rf","xgboost","svm"]
ci_rows=[]; sig_rows=[]
for scheme in folds:
    aucs={m: roc_auc_score(y, oof_proba[(scheme,m)]) for m in learners}
    best=max(aucs, key=aucs.get)
    for m in learners+["dummy"]:
        p=oof_proba[(scheme,m)]
        ra,rlo,rhi=bootstrap_ci(y,p,roc_auc_score)
        pa,plo,phi=bootstrap_ci(y,p,average_precision_score)
        ci_rows.append(dict(split=scheme,model=m,roc_auc=ra,roc_lo=rlo,roc_hi=rhi,
                            pr_auc=pa,pr_lo=plo,pr_hi=phi))
    for m in learners:
        if m==best:
            sig_rows.append(dict(split=scheme,model=m,vs_best=best,delong_p=np.nan,
                                 mcnemar_p=np.nan,note="best")); continue
        _,_,dp=delong_test(y, oof_proba[(scheme,m)], oof_proba[(scheme,best)])
        _,_,mp=mcnemar_test(y, oof_proba[(scheme,m)], oof_proba[(scheme,best)])
        sig_rows.append(dict(split=scheme,model=m,vs_best=best,delong_p=dp,mcnemar_p=mp,note=""))
        # vs dummy
        _,_,dpd=delong_test(y, oof_proba[(scheme,m)], oof_proba[(scheme,"dummy")])

ci=pd.DataFrame(ci_rows); sig=pd.DataFrame(sig_rows)
display(ci.round(3)); display(sig.round(4))

## 7. SHAP feature attribution — all four learners

SHAP quantifies how much each identity feature (position / wt residue / mut residue) pushes a prediction
toward functional or non-functional. This is **interpretability, not a significance test** — it explains
*what the model learned*, so we check it against the EDA finding that the wild-type residue and proline
substitutions carry the most identity signal.

Attribution is computed on the **random-split** first-fold model (attribution is about the learned
function, so one representative split suffices). Trees use exact **TreeSHAP**; logistic regression and SVM
use **KernelSHAP** on a stratified sample of points against a k-means background — the reason the SVM pass
dominates runtime and why this notebook has a Colab twin.

In [ ]:
import shap
rng=np.random.default_rng(SEED)
# stratified sample of points to explain, and a k-means background for kernel methods
tr0,te0=folds[SHAP_SPLIT][0]
expl_idx=np.concatenate([rng.choice(te0[y[te0]==c],
                          size=min(SHAP_N_EXPLAIN//2,(y[te0]==c).sum()), replace=False) for c in [0,1]])
Xexpl=X[expl_idx]
bg=shap.kmeans(X[tr0], SHAP_N_BACKGROUND)

shap_vals={}
for mname in learners:
    m=fitted[(SHAP_SPLIT,mname)]; t0=time.time()
    if mname in ("rf","xgboost"):
        expl=shap.TreeExplainer(m)
        sv=expl.shap_values(Xexpl)
        sv=sv[1] if isinstance(sv,list) else (sv[...,1] if sv.ndim==3 else sv)
    else:
        f=(lambda M: (lambda z: M.predict_proba(z)[:,1]))(m)
        expl=shap.KernelExplainer(f, bg)
        sv=expl.shap_values(Xexpl, nsamples=100, silent=True)
        sv=sv[1] if isinstance(sv,list) else sv
    shap_vals[mname]=np.asarray(sv)
    print(f"{mname:8s} SHAP done in {time.time()-t0:.0f}s  shape={shap_vals[mname].shape}")

In [ ]:
# mean|SHAP| per feature, grouped into position / wt / mut families, per model
def family(f): return f.split("_")[0] if f.startswith("position") else f[:5]
fam=np.array(["position" if f.startswith("position_") else ("wt_aa" if f.startswith("wt_aa_") else "mut_aa")
              for f in feat_names])
imp_rows=[]
for mname,sv in shap_vals.items():
    mabs=np.abs(sv).mean(axis=0)
    for grp in ["position","wt_aa","mut_aa"]:
        imp_rows.append(dict(model=mname, feature_group=grp, mean_abs_shap=float(mabs[fam==grp].sum())))
imp=pd.DataFrame(imp_rows)
display(imp.pivot(index="model",columns="feature_group",values="mean_abs_shap").round(3))

## 8. Figures

Per project convention, figures go to `results/figures/02_traditional_ml_aa_identity_benchmark/` with generic
names. **Confusion matrices are split per model architecture** (one figure each, panels across the three
splits); **ROC, PR, metric bars, utility bars, and SHAP importance are combined** across models.

In [ ]:
# 8a. confusion matrix per model architecture (rows=truth, cols=pred), one figure per learner
for mname in learners:
    fig,axes=plt.subplots(1,3,figsize=(12,3.8))
    for ax,scheme in zip(axes, ["random","modulo","contiguous"]):
        pred=(oof_proba[(scheme,mname)]>=0.5).astype(int)
        cm=confusion_matrix(y,pred,labels=[0,1])
        sns.heatmap(cm,annot=True,fmt="d",cmap="PuBuGn",cbar=False,ax=ax,
                    xticklabels=["non-func","func"],yticklabels=["non-func","func"])
        ax.set_title(f"{scheme}"); ax.set_xlabel("predicted"); ax.set_ylabel("actual")
    fig.suptitle(f"Confusion matrices — {mname}", y=1.03, fontsize=13)
    fig.tight_layout(); fig.savefig(FIGDIR/f"confusion_matrix_{mname}.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# 8b. combined ROC curves (one panel per split, one line per model)
fig,axes=plt.subplots(1,3,figsize=(14,4.4))
for ax,scheme in zip(axes,["random","modulo","contiguous"]):
    for m in learners:
        fpr,tpr,_=roc_curve(y,oof_proba[(scheme,m)]); a=roc_auc_score(y,oof_proba[(scheme,m)])
        ax.plot(fpr,tpr,color=MODEL_COLORS[m],lw=1.6,label=f"{m} ({a:.3f})")
    ax.plot([0,1],[0,1],ls="--",color="grey",lw=1)
    ax.set_title(f"ROC — {scheme}"); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR/"roc_curves.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# 8c. combined PR curves
fig,axes=plt.subplots(1,3,figsize=(14,4.4))
for ax,scheme in zip(axes,["random","modulo","contiguous"]):
    for m in learners:
        pr,rc,_=precision_recall_curve(y,oof_proba[(scheme,m)]); a=average_precision_score(y,oof_proba[(scheme,m)])
        ax.plot(rc,pr,color=MODEL_COLORS[m],lw=1.6,label=f"{m} ({a:.3f})")
    ax.axhline(y.mean(),ls="--",color="grey",lw=1)
    ax.set_title(f"PR — {scheme}"); ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGDIR/"pr_curves.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# 8d. metric bar chart with bootstrap CIs (ROC-AUC), grouped by split
fig,ax=plt.subplots(figsize=(11,4.6))
schemes=["random","modulo","contiguous"]; width=0.18
xbase=np.arange(len(schemes))
for i,m in enumerate(learners):
    vals=[ci[(ci.split==s)&(ci.model==m)].roc_auc.values[0] for s in schemes]
    los=[ci[(ci.split==s)&(ci.model==m)].roc_lo.values[0] for s in schemes]
    his=[ci[(ci.split==s)&(ci.model==m)].roc_hi.values[0] for s in schemes]
    err=[np.array(vals)-np.array(los), np.array(his)-np.array(vals)]
    ax.bar(xbase+i*width, vals, width, yerr=err, capsize=3, color=MODEL_COLORS[m], label=m)
dummy=[ci[(ci.split==s)&(ci.model=="dummy")].roc_auc.values[0] for s in schemes]
ax.axhline(0.5,ls=":",color="grey",lw=1,label="chance")
ax.set_xticks(xbase+1.5*width); ax.set_xticklabels(schemes); ax.set_ylim(0.4,1.0)
ax.set_ylabel("ROC-AUC (95% CI)"); ax.set_title("ROC-AUC by model and split"); ax.legend(fontsize=8,ncol=5)
fig.tight_layout(); fig.savefig(FIGDIR/"metric_bars.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# 8e. utility-score bar chart (mean utility per prediction), grouped by split
fig,ax=plt.subplots(figsize=(11,4.6))
for i,m in enumerate(learners):
    vals=[grid[(grid.split==s)&(grid.model==m)].utility_mean.values[0] for s in schemes]
    errs=[grid[(grid.split==s)&(grid.model==m)].utility_std.values[0] for s in schemes]
    ax.bar(xbase+i*width, vals, width, yerr=errs, capsize=3, color=MODEL_COLORS[m], label=m)
dv=[grid[(grid.split==s)&(grid.model=="dummy")].utility_mean.values[0] for s in schemes]
ax.plot(xbase+1.5*width, dv, "k_", ms=18, label="dummy")
ax.set_xticks(xbase+1.5*width); ax.set_xticklabels(schemes)
ax.set_ylabel("mean utility / prediction"); ax.set_title("Utility score (TP+1, TN+1, FN-0.25, FP-1)")
ax.legend(fontsize=8,ncol=5); fig.tight_layout()
fig.savefig(FIGDIR/"utility_bars.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# 8f. combined SHAP feature-group importance across models
fig,ax=plt.subplots(figsize=(9,4.4))
piv=imp.pivot(index="model",columns="feature_group",values="mean_abs_shap").reindex(learners)
piv=piv[["position","wt_aa","mut_aa"]]
piv.plot(kind="bar", ax=ax, color=[PAL["blue"],PAL["green"],PAL["purple"]], width=0.75)
ax.set_ylabel("sum mean|SHAP| in group"); ax.set_xlabel("")
ax.set_title(f"SHAP feature-group importance ({SHAP_SPLIT} split)")
ax.legend(title="feature group", fontsize=9); plt.xticks(rotation=0)
fig.tight_layout(); fig.savefig(FIGDIR/"shap_importance.pdf", bbox_inches="tight"); plt.show()

## 9. Save metrics tables and model artifacts

Tables (metrics grid, CIs, significance) go to `results/tables/`. Each model is refit on the **full data**
and bundled per `CLAUDE.md` with everything needed to reproduce its input features: the fitted encoder,
feature names and column order, the utility matrix, split metadata, and its OOF metrics.

In [ ]:
# metrics tables
grid.round(4).to_csv(TABLES/f"{NBNAME}_metrics_grid.csv", index=False)
ci.round(4).to_csv(TABLES/f"{NBNAME}_bootstrap_ci.csv", index=False)
sig.round(5).to_csv(TABLES/f"{NBNAME}_significance.csv", index=False)
imp.round(4).to_csv(TABLES/f"{NBNAME}_shap_importance.csv", index=False)
print("wrote 4 tables to", TABLES.relative_to(BASE_DIR))

In [ ]:
# refit each learner on all data and bundle with feature metadata (CLAUDE.md: reusable artifacts)
enc_full = OneHotEncoder(categories=[sorted(df.position.unique()), AA, AA],
                         handle_unknown="ignore", sparse_output=False, dtype=np.float32).fit(df[["position","wt_aa","mut_aa"]])
for mname, mk in make_models().items():
    if mname=="dummy": continue
    m=mk(); m.fit(X, y)
    bundle=dict(model=m, encoder=enc_full, feature_names=list(feat_names),
                feature_order=["position","wt_aa","mut_aa"], utility_matrix=UTIL,
                positive_class="functional(1)", n_splits=N_SPLITS, seed=SEED,
                oof_metrics={s: fold_metrics[(s,mname)].mean().to_dict() for s in folds},
                source="data/processed/traditional_ml_aa_identity/modeling_dataset.parquet")
    joblib.dump(bundle, MODELDIR/f"{mname}.joblib")
print("saved", len(list(MODELDIR.glob('*.joblib'))), "model bundles to", MODELDIR.relative_to(BASE_DIR))

## 10. Summary

- **The AA-identity baseline is not trivial**: on the random split, identity alone already separates
  functional from non-functional variants well (see the metrics grid) — because residue chemistry
  (which residue, to what) carries real signal, as the EDA association tests showed.
- **The split gap is the headline**: performance drops from random → modulo → contiguous. That gap is the
  amount of apparent skill that was position memorization; what remains on the contiguous split is
  transferable substitution chemistry — the honest floor for a region-holdout setting.
- **This is the number to beat.** Any pLLM-feature model must exceed this identity-only baseline —
  especially on the contiguous split — to justify the added complexity. That is the comparison the next
  phase of the project makes.

All metrics, CIs, significance tests, SHAP attributions, figures, and model bundles are written to disk for
the write-up and for reuse.